


##Automation Driving - highway-env

1.   Establising Stable Baselines for highway
2.   Creating better model for the env






In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%pip install highway-env
%pip install stable-baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.2/187.2 kB 4.7 MB/s eta 0:00:00


In [ ]:
import os

os.makedirs("./highway", exist_ok=True)

In [ ]:
%cd /content/drive/MyDrive/cs-272-final-project/highway

/content/drive/MyDrive/cs-272-final-project/custom


In [ ]:
# Ignore warnings
import warnings
warnings.filterwarnings("ignore",category=DeprecationWarning,)
warnings.filterwarnings("ignore", message=".*Kernel._parent_header.*")

In [ ]:
# Learning curve (episodes on X) from Monitor CSVs
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym
import highway_env  # registers env IDs
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

def plot_learning_curve_monitor(monitor_dir: str, out_png: str, smooth_window: int = 50):
    files = sorted(glob.glob(os.path.join(monitor_dir, "**", "*.csv"), recursive=True))
    if not files:
        raise FileNotFoundError(f"No monitor CSVs found under: {monitor_dir}")

    dfs = []
    for f in files:
        try:
            df = pd.read_csv(f, comment="#")
            if "r" in df.columns:
                dfs.append(df[["r"]].copy())
        except Exception:
            pass
    if not dfs:
        raise RuntimeError("Found CSVs but no episodic returns (column 'r').")

    log = pd.concat(dfs, ignore_index=True)
    log["episode"] = np.arange(1, len(log) + 1)

    win = max(1, min(smooth_window, len(log)))
    log["r_smooth"] = log["r"].rolling(win, min_periods=1).mean()

    os.makedirs(os.path.dirname(out_png) or ".", exist_ok=True)
    plt.figure()
    plt.plot(log["episode"], log["r_smooth"])
    plt.xlabel("Training Episodes")
    plt.ylabel("Mean Episodic Return (smoothed)")
    plt.title("Learning Curve — Highway LiDAR (PPO)")
    plt.tight_layout()
    plt.savefig(out_png, dpi=140)
    plt.close()
    print(f"Saved: {out_png}")

In [ ]:
# 500-episode deterministic evaluation violin plot
import os
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from stable_baselines3 import PPO

def evaluate_and_plot_violin(model_path: str, out_png: str,
                             env_id: str = "highway-fast-v0",
                             obs_config: dict = None,
                             n_episodes: int = 500,
                             deterministic: bool = True,
                             seed: int = 0):
    env = gym.make(env_id, config=(obs_config or {}))
    model = PPO.load(model_path)

    returns = []
    for _ in range(n_episodes):
        obs, info = env.reset(seed=seed)
        done = False
        ep_ret = 0.0
        while not done:
            action, _ = model.predict(obs, deterministic=deterministic)
            obs, reward, terminated, truncated, info = env.step(action)
            ep_ret += reward
            done = terminated or truncated
        returns.append(ep_ret)
    env.close()

    os.makedirs(os.path.dirname(out_png) or ".", exist_ok=True)
    plt.figure()
    plt.violinplot([returns], showmeans=True)
    plt.xticks([1], ["PPO"])
    plt.ylabel("Episodic Return (n=500, deterministic)")
    plt.title("Performance Test — Highway GrayScale (PPO)")
    plt.tight_layout()
    plt.savefig(out_png, dpi=140)
    plt.close()
    print(f"Saved: {out_png}")



In [ ]:
import os
import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy


def main():

    BASE_DIR   = os.getcwd()                     # highway/baseline
    MONITOR_DIR = os.path.join(BASE_DIR, "runs_monitor")
    MODEL_DIR   = os.path.join(BASE_DIR, "models")
    PLOT_DIR    = os.path.join(BASE_DIR, "plots")
    os.makedirs(MONITOR_DIR, exist_ok=True)
    os.makedirs(MODEL_DIR,   exist_ok=True)
    os.makedirs(PLOT_DIR,    exist_ok=True)

    # Vectorized training env with monitor logging
    vec_env = make_vec_env(
        "highway-fast-v0",
        n_envs=4,
        seed=42,
        monitor_dir=MONITOR_DIR,
    )

    # PPO model
    model = PPO(
        "MlpPolicy",
        vec_env,
        verbose=1,
        n_steps=2048,
        batch_size=64,
        learning_rate=3e-4,
        ent_coef=0.0,
    )

    # Train (using a multiple of n_steps * n_envs = 2048*4 = 8192)
    total_timesteps = int(2048)
    model.learn(total_timesteps=total_timesteps, progress_bar=False)

    # Save
    model_path = os.path.join(BASE_DIR, "ppo_custom_env")
    model.save(model_path)
    print(f"Model saved to: {model_path}.zip")

    plot_learning_curve_monitor(MONITOR_DIR, os.path.join(PLOT_DIR, "GrayScale_learning.png"))
    evaluate_and_plot_violin(
        model_path + ".zip",
        os.path.join(PLOT_DIR, "GrayScale_violin.png"),
        env_id="highway-fast-v0",
        obs_config=gray_config,
        n_episodes=500,
        deterministic=True,
    )

    # Clean up
    vec_env.close()

if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/gymnasium/envs/registration.py:636: UserWarning: WARN: Overriding environment MyCustomEnv-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


Using cpu device
